In [1]:
import sys
import re
import string
from collections import Counter
import nltk
from nltk.corpus import words as nltk_words
from nltk.corpus import gutenberg
from nltk.corpus import brown

In [2]:
# downloads nltk
nltk.download('words')
nltk.download('brown')
nltk.download('gutenberg')

[nltk_data] Downloading package words to /home/david/nltk_data...
[nltk_data]   Package words is already up-to-date!
[nltk_data] Downloading package brown to /home/david/nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package gutenberg to /home/david/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


True

In [8]:
eng_words = set()
vocab = Counter()
nltk_text = ""

eng_words = set(w.lower() for w in nltk_words.words())
nltk_text += " " + " ".join(brown.words()).lower()
vocab.update([w.lower() for w in brown.words()])

nltk_text += " " + gutenberg.raw().lower()
vocab.update([w.lower() for w in gutenberg.words()])

input_path = '/home/david/Documents/HackerRank-challenges/the-missing-characters/the-missing-characters-testcases/input/input02.txt'

In [ ]:
# --- Clean and normalize texts ---
def clean_text(text):
    # Discard punctuation to allow robust n-gram string matching
    t = re.sub(r'[^a-z\s#]', '', text.lower())
    return re.sub(r'\s+', ' ', t)

# Dictionary mapping standard English letter frequencies (for tie-breaking edge cases)
letter_freq = {
    'e': 13.0, 't': 9.1, 'a': 8.2, 'o': 7.5, 'i': 7.0, 'n': 6.7, 's': 6.3, 'h': 6.1, 'r': 6.0, 
    'd': 4.3, 'l': 4.0, 'c': 2.8, 'u': 2.8, 'm': 2.5, 'w': 2.4, 'f': 2.2, 'g': 2.0, 'y': 2.0, 
    'p': 1.9, 'b': 1.5, 'v': 0.98, 'k': 0.77, 'j': 0.15, 'x': 0.15, 'q': 0.09, 'z': 0.07
}

with open(input_path, 'r', encoding='utf-8') as f:
    content = f.read().strip()
    cleaned_input = clean_text(content)

cleaned_nltk = clean_text(nltk_text)

def find_best_char(candidates, context_pairs):
    """Finds the optimal missing character by string-matching contexts of decreasing size."""
    for lp, rp in context_pairs:
        current_scores = {}
        for c in candidates:
            # Reconstruct string sequence replacing '#' with the candidate letter (case agnostic)
            query = lp + c + rp
            score = cleaned_nltk.count(query)
                
            if score > 0:
               current_scores[c] = score
                    
        if current_scores:
            # Return character with the highest n-gram match score (break ties using standard frequencies)
            return max(current_scores) # max(current_scores, key=lambda x: (current_scores[x], letter_freq.get(x, 0)))
    return None

In [ ]:
# --- Process each missing character '#' ---
# Find the positions of all '#' tags
hashes = [i for i, char in enumerate(cleaned_input) if char == '#']
missing_characters = []

for h_idx in hashes:
    # a. Isolate the target word containing the '#' symbol
    start = h_idx
    while start > 0 and cleaned_input[start - 1] != ' ':
        start -= 1
    end = h_idx
    while end < len(cleaned_input) - 1 and cleaned_input[end + 1] != ' ':
        end += 1

    word_pattern = cleaned_input[start:end + 1]
    local_idx = h_idx - start

    # b. Find valid substitutions purely using dictionary lookups
    valid_candidates = []
    if '#' not in (word_pattern[:local_idx] + word_pattern[local_idx + 1:]):
        for c in string.ascii_lowercase:
            candidate_word = word_pattern[:local_idx] + c + word_pattern[local_idx + 1:]
            if candidate_word in vocab or candidate_word in eng_words:
                valid_candidates.append(c)

    candidates_to_check = valid_candidates if len(valid_candidates) > 0 else list(string.ascii_lowercase)

    # c. Extract contiguous left and right contexts (safely stopping if another '#' is adjacent)
    left_limit = h_idx - 1
    while left_limit >= 0 and cleaned_input[left_limit] != '#':
        left_limit -= 1
    left_context_full = cleaned_input[left_limit + 1:h_idx]

    right_limit = h_idx + 1
    while right_limit < len(cleaned_input) and cleaned_input[right_limit] != '#':
        right_limit += 1
    right_context_full = cleaned_input[h_idx + 1:right_limit]

    # d. Build context pairs of shrinking sizes
    context_pairs = []
    max_len = max(len(left_context_full), len(right_context_full))
    for w in range(min(40, max_len), 0, -1):
        lp = left_context_full[-w:] if w > 0 else ""
        rp = right_context_full[:w] if w > 0 else ""
        pair = (lp, rp)
        if not context_pairs or context_pairs[-1] != pair:
            context_pairs.append(pair)

    # e. Evaluate candidates
    best_char = None

    if len(valid_candidates) == 1:
        best_char = valid_candidates[0]
    else:
        best_char = find_best_char(candidates_to_check, context_pairs)

    if not best_char:
        unigram_scores = {}
        for c in candidates_to_check:
            unigram_scores[c] = cleaned_nltk.count(c)

        if unigram_scores:
            best_char = max(unigram_scores, key=lambda x: (unigram_scores[x], letter_freq.get(x, 0)))
        else:
            best_char = 'e'

    print(best_char)

u
o
s
n
a
y
t
t
y
n
e
e
v
g
t
s
t
d
b
i
i
y
e
c
h
f
t
i
k
u


In [13]:
output_path = '/home/david/Documents/HackerRank-challenges/the-missing-characters/the-missing-characters-testcases/output/output02.txt'

with open(output_path, 'r', encoding='utf-8') as f:
    output = f.read().strip()

output = output.split('\n')

In [20]:
missing = []
for a, b in zip(missing_characters, output):
    missing.append(a != b)

In [23]:
len(missing)

30